<a href="https://colab.research.google.com/github/ayyanarh1/tamil-nadu-school-flood-risk/blob/main/day8_cyclone_wind_hazard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Setup
!pip install geopandas folium pandas numpy matplotlib requests -q

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import requests

print('✅ Day 8 ready!')

✅ Day 8 ready!


In [2]:
# Download STORM cyclone track data for Bay of Bengal
import requests
import pandas as pd
import numpy as np

print('Downloading IBTrACS cyclone track data...')

# IBTrACS — International Best Track Archive for Climate Stewardship
# Free, no login required
url = 'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r00/access/csv/ibtracs.NI.list.v04r00.csv'

r = requests.get(url, timeout=60)

if r.status_code == 200:
    with open('ibtracs_NI.csv', 'wb') as f:
        f.write(r.content)
    print('✅ IBTrACS North Indian Ocean data downloaded!')
else:
    print(f'❌ Failed — status {r.status_code}')

✅ IBTrACS North Indian Ocean data downloaded!


In [3]:
# Load and explore IBTrACS cyclone data
import pandas as pd
import numpy as np

# Load IBTrACS data
# Skip first row (units row)
df_raw = pd.read_csv('ibtracs_NI.csv', skiprows=[1], low_memory=False)

print('=== IBTrACS North Indian Ocean Dataset ===')
print(f'Total records: {len(df_raw)}')
print(f'Columns: {list(df_raw.columns[:10])}')
print(f'Date range: {df_raw.SEASON.min()} to {df_raw.SEASON.max()}')

# Filter to Bay of Bengal cyclones only
# Longitude 76-90 covers Tamil Nadu coast
bol = df_raw[
    (df_raw['LON'].astype(str).str.replace(' ','') != '') &
    (pd.to_numeric(df_raw['LON'], errors='coerce').between(70, 90)) &
    (pd.to_numeric(df_raw['LAT'], errors='coerce').between(5, 25))
].copy()

bol['LON'] = pd.to_numeric(bol['LON'], errors='coerce')
bol['LAT'] = pd.to_numeric(bol['LAT'], errors='coerce')
bol['SEASON'] = pd.to_numeric(bol['SEASON'], errors='coerce')

# Recent cyclones only (2000 onwards)
recent = bol[bol['SEASON'] >= 2000].copy()

print(f'\nBay of Bengal cyclone records (2000+): {len(recent)}')
print(f'Unique storms: {recent.SID.nunique()}')
print(f'Most recent season: {recent.SEASON.max()}')

=== IBTrACS North Indian Ocean Dataset ===
Total records: 60677
Columns: ['SID', 'SEASON', 'NUMBER', 'BASIN', 'SUBBASIN', 'NAME', 'ISO_TIME', 'NATURE', 'LAT', 'LON']
Date range: 1842 to 2024

Bay of Bengal cyclone records (2000+): 4273
Unique storms: 171
Most recent season: 2024


In [4]:
# Calculate cyclone wind hazard per school
import pandas as pd
import numpy as np

# School coordinates
schools = {
    'School Puducherry Border':      [11.93, 79.83],
    'Panchayat School Nagapattinam': [10.76, 79.84],
    'School Ramanathapuram':         [9.37,  78.83],
    'Govt School Cuddalore':         [11.75, 79.75],
    'School Tuticorin':              [8.80,  78.15],
    'School Kanchipuram':            [12.83, 79.70],
    'Govt School Tiruchirappalli':   [10.79, 78.68],
    'Panchayat School Tirunelveli':  [8.71,  77.69],
    'Govt School Thanjavur':         [10.78, 79.13],
    'High School Vellore':           [12.92, 79.13],
    'School Villupuram':             [11.93, 79.49],
    'Govt School Madurai':           [9.93,  78.12],
    'Govt High School Chennai':      [13.08, 80.27],
    'High School Coimbatore':        [11.01, 76.96],
    'Govt School Salem':             [11.65, 78.16],
}

# For each school count cyclone tracks within 200km
# Using Haversine distance formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Get unique storm positions
storm_points = recent[['SID', 'LAT', 'LON', 'SEASON']].dropna().copy()
storm_points['LAT'] = pd.to_numeric(storm_points['LAT'], errors='coerce')
storm_points['LON'] = pd.to_numeric(storm_points['LON'], errors='coerce')
storm_points = storm_points.dropna()

print('Calculating cyclone proximity for each school...')

results = []
for name, (lat, lon) in schools.items():
    # Distance from school to every storm point
    distances = haversine(lat, lon,
                         storm_points['LAT'].values,
                         storm_points['LON'].values)

    # Count unique storms within 200km
    storm_points_near = storm_points[distances <= 200].copy()
    storms_200km = storm_points_near['SID'].nunique()

    # Count unique storms within 100km
    storms_100km = storm_points[distances <= 100]['SID'].nunique()

    # Most recent storm
    if len(storm_points_near) > 0:
        recent_year = int(storm_points_near['SEASON'].max())
    else:
        recent_year = 0

    results.append({
        'school_name': name,
        'storms_200km': storms_200km,
        'storms_100km': storms_100km,
        'recent_storm_year': recent_year
    })

cyclone_df = pd.DataFrame(results)

# Cyclone risk tier
def cyclone_risk(count):
    if count >= 10:   return 'HIGH'
    elif count >= 5:  return 'MEDIUM'
    else:             return 'LOW'

cyclone_df['cyclone_risk'] = cyclone_df['storms_200km'].apply(cyclone_risk)
cyclone_df = cyclone_df.sort_values('storms_200km', ascending=False).reset_index(drop=True)

print('\n=== CYCLONE WIND HAZARD — TAMIL NADU SCHOOLS ===')
print('(Unique storm tracks within 200km, 2000-present)\n')
for _, row in cyclone_df.iterrows():
    print(f"{row['school_name'][:35]:<35} | "
          f"200km: {row['storms_200km']:>3} | "
          f"100km: {row['storms_100km']:>3} | "
          f"Last: {row['recent_storm_year']} | "
          f"Risk: {row['cyclone_risk']}")

Calculating cyclone proximity for each school...

=== CYCLONE WIND HAZARD — TAMIL NADU SCHOOLS ===
(Unique storm tracks within 200km, 2000-present)

Govt High School Chennai            | 200km:  24 | 100km:  12 | Last: 2023 | Risk: HIGH
Govt School Cuddalore               | 200km:  21 | 100km:  11 | Last: 2022 | Risk: HIGH
School Puducherry Border            | 200km:  21 | 100km:  10 | Last: 2022 | Risk: HIGH
School Villupuram                   | 200km:  20 | 100km:  11 | Last: 2022 | Risk: HIGH
Panchayat School Nagapattinam       | 200km:  19 | 100km:   9 | Last: 2022 | Risk: HIGH
School Kanchipuram                  | 200km:  19 | 100km:  11 | Last: 2023 | Risk: HIGH
High School Vellore                 | 200km:  16 | 100km:   9 | Last: 2023 | Risk: HIGH
Govt School Salem                   | 200km:  15 | 100km:   4 | Last: 2021 | Risk: HIGH
Govt School Thanjavur               | 200km:  14 | 100km:   9 | Last: 2020 | Risk: HIGH
Govt School Tiruchirappalli         | 200km:  12 | 100km:  

In [7]:
# Multi-hazard combined score
import pandas as pd

# Flood risk from Day 7 (master scores)
flood_data = {
    'school_name': [
        'School Puducherry Border',
        'Panchayat School Nagapattinam',
        'School Ramanathapuram',
        'Govt School Cuddalore',
        'School Tuticorin',
        'School Kanchipuram',
        'Govt School Tiruchirappalli',
        'Panchayat School Tirunelveli',
        'Govt School Thanjavur',
        'High School Vellore',
        'School Villupuram',
        'Govt School Madurai',
        'Govt High School Chennai',
        'High School Coimbatore',
        'Govt School Salem'
    ],
    'connectivity': [
        'connected', 'none', 'none', 'none',
        'connected', 'none', 'none', 'none',
        'connected', 'connected', 'none',
        'connected', 'connected', 'connected', 'connected'
    ],
    'flood_score': [
        96.7, 56.8, 43.5, 39.8, 39.1,
        33.3, 31.4, 30.7, 26.5, 22.1,
        22.9, 13.0, 13.8, 4.8, 4.9
    ]
}

flood_df = pd.DataFrame(flood_data)

# Merge with cyclone data
multi = flood_df.merge(
    cyclone_df[['school_name', 'storms_200km',
                'storms_100km', 'cyclone_risk']],
    on='school_name'
)

# Normalise cyclone score 0-100
max_storms = multi['storms_200km'].max()
multi['cyclone_norm'] = (
    multi['storms_200km'] / max_storms * 100
).round(1)

# Multi-hazard score
# Flood 60% + Cyclone 40%
multi['multi_hazard_score'] = (
    0.60 * multi['flood_score'] +
    0.40 * multi['cyclone_norm']
).round(1)

# Connectivity penalty
multi['conn_penalty'] = multi['connectivity'].apply(
    lambda x: 15 if x == 'none' else 0
)
multi['final_score'] = (
    multi['multi_hazard_score'] + multi['conn_penalty']
).round(1)

# Risk tier
def multi_risk(score):
    if score >= 50:   return 'CRITICAL'
    elif score >= 30: return 'HIGH'
    elif score >= 15: return 'MEDIUM'
    else:             return 'LOW'

multi['final_risk'] = multi['final_score'].apply(multi_risk)
multi = multi.sort_values(
    'final_score', ascending=False
).reset_index(drop=True)
multi['rank'] = multi.index + 1

# Print results
print('=== MULTI-HAZARD RISK — FLOOD + CYCLONE ===')
print('Flood(60%) + Cyclone(40%) + Connectivity penalty\n')
print(f"{'#':<3} {'School':<32} {'Conn':<5} "
      f"{'Flood':>6} {'Cyc':>5} {'Score':>7} {'Risk'}")
print('-' * 68)
for _, row in multi.iterrows():
    conn = 'No' if row['connectivity'] == 'none' else 'Yes'
    print(
        f"#{int(row['rank']):<2} "
        f"{row['school_name'][:31]:<32} "
        f"{conn:<5} "
        f"{row['flood_score']:>5.1f} "
        f"{row['cyclone_norm']:>5.1f} "
        f"{row['final_score']:>6.1f} "
        f"{row['final_risk']}"
    )

=== MULTI-HAZARD RISK — FLOOD + CYCLONE ===
Flood(60%) + Cyclone(40%) + Connectivity penalty

#   School                           Conn   Flood   Cyc   Score Risk
--------------------------------------------------------------------
#1  School Puducherry Border         Yes    96.7  87.5   93.0 CRITICAL
#2  Panchayat School Nagapattinam    No     56.8  79.2   80.8 CRITICAL
#3  Govt School Cuddalore            No     39.8  87.5   73.9 CRITICAL
#4  School Kanchipuram               No     33.3  79.2   66.7 CRITICAL
#5  School Villupuram                No     22.9  83.3   62.1 CRITICAL
#6  School Ramanathapuram            No     43.5  45.8   59.4 CRITICAL
#7  Govt School Tiruchirappalli      No     31.4  50.0   53.8 CRITICAL
#8  Panchayat School Tirunelveli     No     30.7  37.5   48.4 HIGH
#9  Govt High School Chennai         Yes    13.8 100.0   48.3 HIGH
#10 High School Vellore              Yes    22.1  66.7   39.9 HIGH
#11 Govt School Thanjavur            Yes    26.5  58.3   39.2 HIGH
#12

In [8]:
# Multi-hazard interactive map
import folium

# Coordinates
coords = {
    'School Puducherry Border':      [11.93, 79.83],
    'Panchayat School Nagapattinam': [10.76, 79.84],
    'School Ramanathapuram':         [9.37,  78.83],
    'Govt School Cuddalore':         [11.75, 79.75],
    'School Tuticorin':              [8.80,  78.15],
    'School Kanchipuram':            [12.83, 79.70],
    'Govt School Tiruchirappalli':   [10.79, 78.68],
    'Panchayat School Tirunelveli':  [8.71,  77.69],
    'Govt School Thanjavur':         [10.78, 79.13],
    'High School Vellore':           [12.92, 79.13],
    'School Villupuram':             [11.93, 79.49],
    'Govt School Madurai':           [9.93,  78.12],
    'Govt High School Chennai':      [13.08, 80.27],
    'High School Coimbatore':        [11.01, 76.96],
    'Govt School Salem':             [11.65, 78.16],
}

multi['latitude']  = multi['school_name'].map(lambda x: coords[x][0])
multi['longitude'] = multi['school_name'].map(lambda x: coords[x][1])

# Build map
m = folium.Map(
    location=[10.5, 78.5],
    zoom_start=7,
    tiles='CartoDB positron'
)

def get_color(risk):
    return {
        'CRITICAL': '#cc0000',
        'HIGH':     '#ff6600',
        'MEDIUM':   '#ffaa00',
        'LOW':      '#2d8a4e'
    }.get(risk, 'gray')

def get_radius(score):
    if score >= 50:  return 20
    elif score >= 30: return 15
    elif score >= 15: return 10
    else:             return 7

for _, row in multi.iterrows():
    color  = get_color(row['final_risk'])
    radius = get_radius(row['final_score'])
    conn   = '⚠ No connectivity' if row['connectivity'] == 'none' else '✅ Connected'

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        tooltip=f"#{int(row['rank'])} {row['school_name']}",
        popup=folium.Popup(
            f"<b>#{int(row['rank'])} {row['school_name']}</b><br>"
            f"<hr style='margin:4px 0'>"
            f"🌊 Flood score: {row['flood_score']}<br>"
            f"🌪 Cyclone score: {row['cyclone_norm']}<br>"
            f"📊 Multi-hazard: {row['final_score']}<br>"
            f"📡 {conn}<br>"
            f"<hr style='margin:4px 0'>"
            f"🎯 Risk: <b>{row['final_risk']}</b>",
            max_width=260
        )
    ).add_to(m)

    folium.Marker(
        location=[row['latitude'], row['longitude']],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px;font-weight:bold;'
                 f'color:white;text-align:center;margin-top:4px">'
                 f'#{int(row["rank"])}</div>',
            icon_size=(20, 20),
            icon_anchor=(10, 10)
        )
    ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;
     background:white;padding:14px;border-radius:10px;
     border:1px solid #ccc;font-size:12px;z-index:1000;
     box-shadow:2px 2px 6px rgba(0,0,0,0.15)">
     <b>Multi-Hazard Risk</b><br>
     <b>Flood + Cyclone</b><br><br>
     <span style='color:#cc0000;font-size:16px'>●</span> Critical<br>
     <span style='color:#ff6600;font-size:16px'>●</span> High<br>
     <span style='color:#ffaa00;font-size:16px'>●</span> Medium<br>
     <span style='color:#2d8a4e;font-size:16px'>●</span> Low<br><br>
     <i>Size = risk magnitude</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save('tamil_nadu_multihazard_map.html')
print('✅ Multi-hazard map saved!')
display(m)

✅ Multi-hazard map saved!


In [9]:
# Save all Day 8 outputs
from google.colab import files
import pandas as pd

# Save multi-hazard scores
multi.to_csv('tamil_nadu_multihazard_scores.csv', index=False)
cyclone_df.to_csv('tamil_nadu_cyclone_hazard.csv', index=False)

# Download all files
files.download('tamil_nadu_multihazard_map.html')
files.download('tamil_nadu_multihazard_scores.csv')
files.download('tamil_nadu_cyclone_hazard.csv')

print('✅ All files downloaded!')
print()
print('=== DAY 8 SUMMARY ===')
print(f"Total cyclone records analysed: {len(recent)}")
print(f"CRITICAL schools (flood only):  2")
print(f"CRITICAL schools (multi-hazard): {len(multi[multi.final_risk == 'CRITICAL'])}")
print(f"HIGH schools: {len(multi[multi.final_risk == 'HIGH'])}")
print(f"Most exposed school: {multi.iloc[0]['school_name']}")
print(f"Highest cyclone exposure: {multi.nlargest(1, 'cyclone_norm').iloc[0]['school_name']}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files downloaded!

=== DAY 8 SUMMARY ===
Total cyclone records analysed: 4273
CRITICAL schools (flood only):  2
CRITICAL schools (multi-hazard): 7
HIGH schools: 5
Most exposed school: School Puducherry Border
Highest cyclone exposure: Govt High School Chennai
